# 04. Bảng và biểu đồ cho slide

Tạo bảng kết quả và các file PNG trong `outputs/results`. Biểu đồ nào thiếu dữ liệu sẽ được bỏ qua, nên có thể chạy lại sau mỗi lượt.

In [ ]:
import os, subprocess, sys
from pathlib import Path

# Đổi thành địa chỉ repo của bạn. Repo private thì thêm secret GITHUB_TOKEN trên Colab.
REPO_URL = "https://github.com/TEN-TAI-KHOAN/vihsd-lora.git"

def github_token():
    if os.environ.get("GITHUB_TOKEN"):
        return os.environ["GITHUB_TOKEN"]
    try:
        from google.colab import userdata
        return userdata.get("GITHUB_TOKEN")
    except Exception:
        return None

def find_repo():
    places = [Path.cwd(), *Path.cwd().parents, Path("/content/vihsd-lora"), Path("/kaggle/working/vihsd-lora")]
    places += list(Path("/kaggle/input").glob("*")) + list(Path("/kaggle/input").glob("*/vihsd-lora"))
    return next((place for place in places if (place / "src" / "vihsd_lora").is_dir()), None)

repo = find_repo()
if repo is None:
    if "TEN-TAI-KHOAN" in REPO_URL:
        raise ValueError("Hãy sửa REPO_URL thành địa chỉ repo GitHub của bạn rồi chạy lại ô này.")
    repo = Path("/content" if Path("/content").exists() else "/kaggle/working") / "vihsd-lora"
    token = github_token()
    url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL
    subprocess.run(["git", "clone", "--depth", "1", url, str(repo)], check=True)
elif (repo / ".git").exists():
    subprocess.run(["git", "-C", str(repo), "pull", "--ff-only", "-q"], check=False)
if not os.environ.get("VIHSD_SKIP_INSTALL"):
    # Colab cài sẵn torchao bản cũ, peft sẽ báo lỗi khi gắn LoRA. Repo không dùng torchao nên gỡ đi.
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(repo / "requirements.txt")], check=True)
sys.path.insert(0, str(repo / "src"))
print("Repo:", repo)

In [ ]:
from vihsd_lora.config import Settings
from vihsd_lora.env import default_output_root, describe_gpu, detect_platform, free_space_gb, load_secrets

print("Nền tảng:", detect_platform())
print("GPU:", describe_gpu())
print("Secret đã có:", load_secrets())

settings = Settings.from_env(output_root=default_output_root())
free_gb = free_space_gb(settings.output_root)
print("Kết quả lưu tại:", settings.output_root)
print(f"Dung lượng còn trống: {free_gb} GB")
if free_gb < 6:
    print("Cảnh báo: còn dưới 6GB. Lượt full fine-tune cần khoảng 3GB, hãy dọn Drive và làm trống thùng rác.")

## Bảng kết quả

Gồm 13 dòng đã công bố cùng quy trình, mốc luôn đoán CLEAN và các lượt đã chấm test.

In [ ]:
from vihsd_lora.report import export_tables, main_table

display(main_table(settings))
print(export_tables(settings))

## Biểu đồ

In [ ]:
from IPython.display import Image, display as show
from vihsd_lora.plots import make_all

figures = make_all(settings)
for figure in figures:
    print(figure.name)
    show(Image(str(figure), width=760))

## Đưa biểu đồ lên W&B

Tạo run `slide-figures` để cả nhóm xem chung.

In [ ]:
from vihsd_lora.report import log_figures_to_wandb

log_figures_to_wandb(figures, settings)